<a href="https://colab.research.google.com/github/Charvi-M/BERTImp/blob/main/BERTimplementation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch
from transformers import BertTokenizer, BertModel
from scipy.spatial.distance import cosine

In [2]:
"""Now we load the tokenizer to break down sentences into tokens so that
the BERT model uses them for processing."""
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
#we are loading a pretrained BERT tokenizer with the configuration of
#bert-base-uncased, and assigning it to the variable tokenizer
#BertTokenizer: A class from Hugging Face's transformers library that handles
# tokenizing text for BERT models.
#.from_pretrained('bert-base-uncased'):This loads the pretrained tokenizer
#weights and vocabulary from the bert-base-uncased model,
#bert-base: means 12-layer (base) version of BERT.
#uncased: all text will be converted to lowercase, and it ignores case diffrence

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [3]:
txt = "I need to visit the financial bank tomorrow; after that, we'll set up a"
txt+="tent by the river bank, just across from the bank building"
txt+="where my friend works."

In [4]:
"""
Now we need to add two tokens called cls and sep in the begining and end
of txt, respectively.
"""
wrangled_txt = '[CLS] ' + txt + ' [SEP]'

#tokenization
tokenized_txt = tokenizer.tokenize(wrangled_txt)

print(tokenized_txt)

['[CLS]', 'i', 'need', 'to', 'visit', 'the', 'financial', 'bank', 'tomorrow', ';', 'after', 'that', ',', 'we', "'", 'll', 'set', 'up', 'ate', '##nt', 'by', 'the', 'river', 'bank', ',', 'just', 'across', 'from', 'the', 'bank', 'building', '##w', '##her', '##e', 'my', 'friend', 'works', '.', '[SEP]']


In [5]:
#Next, we convert each token to its corresponding index in BERT’s vocabulary.
#get the ids of the tokens
ids_tokens = tokenizer.convert_tokens_to_ids(tokenized_txt)

#Display the tokens
for t in zip(tokenized_txt, ids_tokens):
    print('{:<12} {:>8,}'.format(t[0], t[1]))

[CLS]             101
i               1,045
need            2,342
to              2,000
visit           3,942
the             1,996
financial       3,361
bank            2,924
tomorrow        4,826
;               1,025
after           2,044
that            2,008
,               1,010
we              2,057
'               1,005
ll              2,222
set             2,275
up              2,039
ate             8,823
##nt            3,372
by              2,011
the             1,996
river           2,314
bank            2,924
,               1,010
just            2,074
across          2,408
from            2,013
the             1,996
bank            2,924
building        2,311
##w             2,860
##her           5,886
##e             2,063
my              2,026
friend          2,767
works           2,573
.               1,012
[SEP]             102


In [6]:
#For each token in tokenized_text, we need to indicate whether it belongs to
#the first sentence (represented by 0s) or the second (represented by 1s).
#because we have only one sentence we only need a vector of 1s,
# marking all tokens as part of a single sentence.
segments_ids = [1] * len(tokenized_txt)
#Convert the token IDs and segment IDs into tensors.
#This step because BERT expects PyTorch tensors as input not lists or arrays.
token_tensor = torch.tensor([ids_tokens])
segment_tensor = torch.tensor([segments_ids])

In [7]:
#Loading model from hugging face with the weights
model = BertModel.from_pretrained('bert-base-uncased', output_hidden_states=True, return_dict = True)
# Put the model in "evaluation" mode, meaning feed-forward operation.
model.eval()
#BertModel.from_pretrained('bert-base-uncased', output_hidden_states=True, return_dict = True)
#Loads a pre-trained BERT model (12-layer, 110M parameters) from Hugging Face’s model hub.
#Includes learned weights trained on English Wikipedia + BooksCorpus.
#Enables access to hidden states from all layers, not just the final output.
# output_hidden_states=True: Tells BERT to return the hidden states from all 13
# layers (embedding layer + 12 transformer layers).
#Each layer gives a tensor of shape: (batch_size, sequence_length, hidden_size).


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

BertModel(
  (embeddings): BertEmbeddings(
    (word_embeddings): Embedding(30522, 768, padding_idx=0)
    (position_embeddings): Embedding(512, 768)
    (token_type_embeddings): Embedding(2, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): BertEncoder(
    (layer): ModuleList(
      (0-11): 12 x BertLayer(
        (attention): BertAttention(
          (self): BertSdpaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): BertSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
            (dropout): Dropout(p=0.1, inplace=False

In [8]:
#Compute the output
with torch.no_grad():
    outputs = model(token_tensor, segment_tensor)

hidden_states = outputs.hidden_states

In [9]:
#Examining Outputs in detail
#The first one is initial embeddings
print ("Number of layers:", len(hidden_states))
layer_ptr = 0

print ("Number of batches:", len(hidden_states[layer_ptr]))
batch_ptr = 0

print ("Number of tokens:", len(hidden_states[layer_ptr][batch_ptr]))
token_ptr = 0

print ("Number of hidden units:", len(hidden_states[layer_ptr][batch_ptr][token_ptr]))

Number of layers: 13
Number of batches: 1
Number of tokens: 39
Number of hidden units: 768


In [10]:
#Concatenate all the layers
token_embeddings = torch.stack(hidden_states, dim=0)

#remove the batch dimension
token_embeddings = torch.squeeze(token_embeddings, dim=1)
print(token_embeddings.shape)

torch.Size([13, 39, 768])


In [11]:
#Concatenate all the layers
token_embeddings = torch.stack(hidden_states, dim=0)

#remove the batch dimension
token_embeddings = torch.squeeze(token_embeddings, dim=1)
print(token_embeddings.shape)

#We now have a single tensor that gives us:

#13 representations for each token, showing how BERT processes it across layers.

#We can do things like:

#Average the last 4 layers (a common practice for sentence embeddings)

#Visualize how a token's meaning evolves through layers

#Extract features from a specific layer

torch.Size([13, 39, 768])


In [12]:
#Swap the dimensions so that the word embeddings generated from the layers are grouped together.
# Swap dimensions 0 and 1 so that each word contains the 13 layer hidden states
token_embeddings = token_embeddings.permute(1,0,2)

token_embeddings.size()

torch.Size([39, 13, 768])

In [13]:
#Creating word vectors from the hidden states by summing the embeddings of the last four layers.
#sum the last four layers
token_vectors_sum = []

# token_embeddings is a [35 x 13 x 768] tensor.

# For each token in the sentence...
for token in token_embeddings:

    # `token` is a [12 x 768] tensor

    # Sum the vectors from the last four layers.
    sum_vector = torch.sum(token[-4:], dim=0)

    # Use `sum_vec` to represent `token`.
    token_vectors_sum.append(sum_vector)

print ('Shape is: %d x %d' % (len(token_vectors_sum), len(token_vectors_sum[0])))

Shape is: 39 x 768


In [14]:
#Displaying the index of the word, as we need it to compare the similarity.
for i, t in enumerate(tokenized_txt):
  print (i, t)

0 [CLS]
1 i
2 need
3 to
4 visit
5 the
6 financial
7 bank
8 tomorrow
9 ;
10 after
11 that
12 ,
13 we
14 '
15 ll
16 set
17 up
18 ate
19 ##nt
20 by
21 the
22 river
23 bank
24 ,
25 just
26 across
27 from
28 the
29 bank
30 building
31 ##w
32 ##her
33 ##e
34 my
35 friend
36 works
37 .
38 [SEP]


In [15]:
#compare the word bank in 7, 23, and 29
#txt = "I need to visit the financial bank tomorrow; after that, we'll set up a tent by the river bank, just across from the bank building where my friend works."

same_bank_word = 1 - cosine(token_vectors_sum[7], token_vectors_sum[29])
diff_bank_word1 = 1 - cosine(token_vectors_sum[7], token_vectors_sum[23])
diff_bank_word2 = 1 - cosine(token_vectors_sum[23], token_vectors_sum[29])

print('Vector similarity for  *similar*  meanings:  %.2f' % same_bank_word)
print('Vector similarity for *different* meanings:  %.2f' % diff_bank_word1)
print('Vector similarity for *different* meanings:  %.2f' % diff_bank_word2)

Vector similarity for  *similar*  meanings:  0.79
Vector similarity for *different* meanings:  0.69
Vector similarity for *different* meanings:  0.71
